### python code

In [1]:
import scanpy as sc
import pandas as pd
import decoupler as dc

In [2]:
Meyloid_data = sc.read_h5ad('/media/NaturalPopulationCohort/CIMA数据文件-最新/CIMA_RNA_Myeloid_754770cells_36326genes_compressed_20260105.h5ad')

In [3]:
DC_data = Meyloid_data[Meyloid_data.obs['cell_type_l4'] == 'DC2_CD1C']

In [4]:
def preprocess_RNA_data(adata_RNA):
    for celltype in adata_RNA.obs['cell_type_l4'].unique():
        print(celltype)
        #分小类
        adata_RNA_celltype = adata_RNA[adata_RNA.obs['cell_type_l4'] == celltype]

        #算伪Bulk
        pdata = dc.get_pseudobulk(
            adata_RNA_celltype,
            sample_col='sample',
            groups_col=None,
            mode='mean',
            min_cells=10,
            min_counts=0,
            min_prop=0,
            min_smpls=0)
        
        #生成伪bulk矩阵
        pseudo_matrix = pd.DataFrame(pdata.X)
        pseudo_matrix.columns = pdata.var_names
        pseudo_matrix.index = pdata.obs.index
        pseudo_matrix = pseudo_matrix.loc[:,adata_RNA_celltype.var_names[adata_RNA_celltype.var_names.isin(pdata.var_names)]]

        #只选取在90%的样本中都表达的特征
        non_zero_ratio = (pseudo_matrix != 0).mean()
        columns_to_keep = non_zero_ratio[non_zero_ratio >= 0.9].index
        pseudo_matrix = pseudo_matrix[columns_to_keep]

        # 计算每列的均值和标准差
        means = pseudo_matrix.mean()
        stds = pseudo_matrix.std()
        # 计算变异系数
        cv = (stds / means).abs() * 100
        # 按变异系数从大到小排序
        sorted_columns = cv.sort_values(ascending=False).index
        # 选取变异系数最高的前2000列
        top_2000_columns = sorted_columns[:min(2000,pseudo_matrix.shape[1])]
        # 获取前2000列的子数据框
        top_CV_2000_pseudo_matrix = pseudo_matrix[top_2000_columns]

        if pseudo_matrix.shape[1] > 0:
            pseudo_matrix.to_csv(f'//media/scPBMC2_AnalysisDisk1/CIMA_DC_analysis/{celltype}.csv')
            top_CV_2000_pseudo_matrix.to_csv(f'/media/scPBMC2_AnalysisDisk1/CIMA_DC_analysis/{celltype}_topCV2000.csv')

In [5]:
preprocess_RNA_data(DC_data)

DC2_CD1C


### R代码

In [83]:
topcv2000 <- read.csv('/media/scPBMC2_AnalysisDisk1/CIMA_DC_analysis/DC2_CD1C_topCV2000.csv',row.names = 1)

genelist <- colnames(topcv2000)

sample_meta <- read.csv('/media/scPBMC2_AnalysisDisk1/CIMA_DC_analysis/CIMA_Sample_Metadata.csv',row.names = 1)

sample_meta <- sample_meta[,c('Age','sex')]

topcv2000_withmeta <- merge(
  sample_meta,
  topcv2000,
  by = "row.names",
  all = FALSE
)

rownames(topcv2000_withmeta) <- topcv2000_withmeta$Row.names
topcv2000_withmeta$Row.names <- NULL

In [82]:
library(dplyr)
library(parallel)

In [90]:
# -------------------------------
# 1️⃣ 定义分析函数（加入整体回归）
# -------------------------------
analyze_gene_effects <- function(gene_vector, metadata){
  df <- metadata
  df$expression <- gene_vector
  
  # 交互作用分析
  fit_int <- lm(expression ~ Age * sex, data = df)
  anova_int <- anova(fit_int)
  
  # 性别分层分析
  male_fit <- lm(expression ~ Age, data = subset(df, sex == "Male"))
  female_fit <- lm(expression ~ Age, data = subset(df, sex == "Female"))
  
  # 全体样本线性回归
  overall_fit <- lm(expression ~ Age, data = df)
  
  # 返回结果
  list(
    interaction = data.frame(
      Age_sex_interaction_F = anova_int["Age:sex", "F value"],
      Age_sex_interaction_p = anova_int["Age:sex", "Pr(>F)"]
    ),
    male = data.frame(
      Age_beta = coef(male_fit)["Age"],
      Age_p = summary(male_fit)$coefficients["Age", "Pr(>|t|)"],
      n_male = nrow(subset(df, sex == "Male"))
    ),
    female = data.frame(
      Age_beta = coef(female_fit)["Age"],
      Age_p = summary(female_fit)$coefficients["Age", "Pr(>|t|)"],
      n_female = nrow(subset(df, sex == "Female"))
    ),
    overall = data.frame(
      Age_beta = coef(overall_fit)["Age"],
      Age_p = summary(overall_fit)$coefficients["Age", "Pr(>|t|)"],
      n_total = nrow(df)
    )
  )
}

# -------------------------------
# 2️⃣ 并行处理所有基因
# -------------------------------
genes <- as.character(genelist)
n_cores <- 50

all_gene_results <- setNames(
  mclapply(genes, function(gene){
    analyze_gene_effects(topcv2000_withmeta[[gene]], topcv2000_withmeta)
  }, mc.cores = n_cores),
  genes
)

# -------------------------------
# 3️⃣ 汇总成大表（加 overall 列）
# -------------------------------
final_gene_results <- do.call(rbind, lapply(names(all_gene_results), function(gene){
  data.frame(
    gene = gene,
    interaction_F = all_gene_results[[gene]]$interaction$Age_sex_interaction_F,
    interaction_p = all_gene_results[[gene]]$interaction$Age_sex_interaction_p,
    male_age_beta = all_gene_results[[gene]]$male$Age_beta,
    male_p = all_gene_results[[gene]]$male$Age_p,
    male_n = all_gene_results[[gene]]$male$n_male,
    female_age_beta = all_gene_results[[gene]]$female$Age_beta,
    female_p = all_gene_results[[gene]]$female$Age_p,
    female_n = all_gene_results[[gene]]$female$n_female,
    overall_age_beta = all_gene_results[[gene]]$overall$Age_beta,
    overall_p = all_gene_results[[gene]]$overall$Age_p,
    overall_n = all_gene_results[[gene]]$overall$n_total
  )
}))

In [96]:
sum(final_gene_results$overall_p < 0.05)

[1] 315

In [97]:
sum(final_gene_results$overall_p < 0.05 &  final_gene_results$interaction_p < 0.05)

[1] 16

In [99]:
write.csv(final_gene_results,file = '/media/scPBMC2_AnalysisDisk1/CIMA_DC_analysis/DC2_CD1C_RNA_corr_with_age.csv')